# DECODE YOLO Training (Colab)
This notebook is prepared to run the DECODE dataset pipeline in Google Colab.
It clones your repository, installs dependencies, helps you upload or mount raw videos, extracts frames, samples frames for labeling, and runs YOLO training using `ultralytics`.

In [ ]:
# Clone the repository (change URL if needed)
!git clone https://github.com/Mithilessh2010/RoboScoutAI.git /content/RoboScoutAI
%cd /content/RoboScoutAI
!git pull origin main || true

In [ ]:
# Install Python deps for the dataset pipeline (and optional training libs)
!pip install -r requirements-decode.txt --quiet
!pip install ultralytics --quiet

## Upload or mount raw videos
Choose one approach: upload files directly (suitable for small sets) or mount Google Drive if videos are large.

In [ ]:
# Option A: mount Google Drive and copy videos into the repo folder
from google.colab import drive
drive.mount('/content/drive')
# Example: copy a folder from Drive to the repo raw-videos location
# !cp -r /content/drive/MyDrive/path-to-videos/* decode-training/raw-videos/unsorted/
print('Drive mounted. Copy videos into decode-training/raw-videos/unsorted and then run the next cells.')

In [ ]:
# Option B: upload files directly via the Colab file picker (for a few small videos)
# Uncomment and run if you prefer this method. Uploaded files will appear in /content and you should move them into the repo folder.
# from google.colab import files
# uploaded = files.upload()
# import shutil, os
# os.makedirs('decode-training/raw-videos/unsorted', exist_ok=True)
# for name in uploaded:
#     shutil.move(name, f'decode-training/raw-videos/unsorted/{name}')
# print('Uploaded files moved to decode-training/raw-videos/unsorted')

## Run contact sheet generation, frame extraction, and sampling
Make sure your videos are in `decode-training/raw-videos/unsorted/` before running these cells.

In [ ]:
# Create expected folders (safe to run multiple times)
!mkdir -p decode-training/raw-videos/unsorted
!mkdir -p decode-training/extracted-frames
!mkdir -p decode-training/sampled-frames
!mkdir -p decode-training/reports/contact-sheets

In [ ]:
# Generate contact sheets (samples at 10,30,50,70,90%)
!python scripts/decode/create_video_contact_sheets.py --src decode-training/raw-videos/unsorted --out decode-training/reports/contact-sheets

In [ ]:
# Extract frames at 2 FPS (adjust with --fps). This will write per-video subfolders under decode-training/extracted-frames/
!python scripts/decode/extract_frames.py --src decode-training/raw-videos/unsorted --out decode-training/extracted-frames --fps 2

In [ ]:
# Sample frames for labeling (per-video sample size adjustable)
!python scripts/decode/sample_frames.py --extracted decode-training/extracted-frames --out decode-training/sampled-frames --per-video 20

## Labeling
Now download `decode-training/sampled-frames/` (or point CVAT/Roboflow at it) and label frames using YOLO format.
- Use CVAT to create a project and upload the sampled frames. Export as YOLO annotations.
- Or create a Roboflow project, upload sampled frames, annotate, and export YOLO.
After exporting YOLO: place images and `.txt` label files into `decode-training/labeled-dataset/images/{train,val,test}` and `decode-training/labeled-dataset/labels/{train,val,test}`. Update `decode-training/labeled-dataset/data.yaml` if paths differ.

In [ ]:
# Validate the YOLO dataset (run after you've placed labeled splits)
!python scripts/decode/validate_yolo_dataset.py --data decode-training/labeled-dataset/data.yaml

## Training
If you have labels and want to train in Colab, run the training cell below. The default model is `yolov8n.pt` for a quick first run. Adjust `epochs` and `batch` as needed.

In [ ]:
# Example training cell using ultralytics (this runs in Python, not shell)
from ultralytics import YOLO
model = YOLO('yolov8n.pt')  # small model for first tests
# train: change epochs/batch as needed
model.train(data='decode-training/labeled-dataset/data.yaml', epochs=50, batch=8, imgsz=640)
print('Training finished. Check runs/train/ for results and best.pt')

In [ ]:
# Copy best.pt to Google Drive for safekeeping (adjust destination)
!mkdir -p /content/drive/MyDrive/DECODE_trained_models || true
# find best.pt and copy it
import glob, shutil, os
candidates = glob.glob('runs/train/**/best.pt', recursive=True)
if candidates:
    dst = '/content/drive/MyDrive/DECODE_trained_models/best.pt'
    shutil.copy(candidates[0], dst)
    print('Copied', candidates[0], 'to', dst)
else:
    print('No best.pt found in runs/train yet.')

## Quick evaluation and prediction
After training and copying `best.pt`, run `scripts/decode/evaluate_yolo_decode.py` to validate the model, then use `scripts/decode/predict_video_decode.py` on a sample video.

In [ ]:
# Example: run prediction on a video file and save JSON outputs to decode-training/predictions
# Adjust the path to a video that exists in the workspace.
!python scripts/decode/predict_video_decode.py decode-training/raw-videos/unsorted/example.mp4 --model decode-training/trained-models/best.pt --out decode-training/predictions || true

---
If you want, I can now run the extraction and contact-sheet generation on the 26 videos you attached and prepare a zip of sampled frames ready for upload to CVAT or Roboflow.
Tell me whether to proceed with running extraction and packaging the sampled frames.